# Component Contribution Study

In [0]:
%run ../config/config

In [0]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as Fn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import mlflow
from pyspark.sql import functions as F
from datetime import datetime
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType, IntegerType,
    DoubleType, LongType, TimestampType,
)
from sklearn.metrics import f1_score, accuracy_score

#### 1. Load flat data & create windows

In [0]:
ml_features_df = spark.table(tables["ml_features"])

feature_cols = [c for c in ml_feature_columns if c in ml_features_df.columns]
print(f"Feature columns: {len(feature_cols)}")

tmp_spark = "/tmp/wyckoff_ml_features_07"   # dbfs:/tmp/...
tmp_local = "/dbfs/tmp/wyckoff_ml_features_07"  # pandas reads FUSE mount
dbutils.fs.rm(tmp_spark, recurse=True)

for split in ["train", "val", "test"]:
    ml_features_df.filter(F.col("split") == split) \
        .write.mode("overwrite").parquet(f"{tmp_spark}/{split}")
    print(f"  Written {split} parquet")

train_pdf = pd.read_parquet(f"{tmp_local}/train").sort_values(["symbol", "date"]).reset_index(drop=True)
val_pdf = pd.read_parquet(f"{tmp_local}/val").sort_values(["symbol", "date"]).reset_index(drop=True)
test_pdf = pd.read_parquet(f"{tmp_local}/test").sort_values(["symbol", "date"]).reset_index(drop=True)

dbutils.fs.rm(tmp_spark, recurse=True)
print(f"  Train: {len(train_pdf):,}  Val: {len(val_pdf):,}  Test: {len(test_pdf):,}")

In [0]:
def create_windows(pdf, feature_cols, window_size, stride, pl_threshold):
    """
    Create sliding windows from flat per-day data.
    Per-window z-score normalization. Filters out price-limit-heavy windows.
    Returns: features (N, W, F), labels (N,), symbols (N,), dates (N,) 
    """
    n_feat = len(feature_cols)

    # upper-bound estimate for pre-allocation
    max_windows = sum(
        max(0, (len(g) - window_size) // stride + 1)
        for _, g in pdf.groupby("symbol", sort=False)
    )
    if max_windows == 0:
        return np.array([]), np.array([]), [], []

    features = np.empty((max_windows, window_size, n_feat), dtype=np.float32)
    labels = np.empty(max_windows, dtype=np.int64)
    symbols_out, dates_out = [], []
    idx = 0

    for sym, group in pdf.groupby("symbol", sort=False):
        group = group.sort_values("date").reset_index(drop=True)
        n = len(group)
        if n < window_size:
            continue

        feat = group[feature_cols].values.astype(np.float32)
        feat = np.where(np.isinf(feat), np.nan, feat)
        # forward fill then fill remaining NaN with 0
        feat_df = pd.DataFrame(feat)
        feat = feat_df.ffill().fillna(0.0).values

        labs = group["phase_label"].values
        dts = group["date"].values
        pl = group["price_limit_hit"].values if "price_limit_hit" in group.columns else np.zeros(n)

        for i in range(window_size - 1, n, stride):
            s = i - window_size + 1

            # Skip if too many price-limit-hit days in window
            if np.nansum(pl[s:i+1]) / window_size > pl_threshold:
                continue

            window = feat[s:i+1]

            # Per-window z-score normalization
            mu = np.nanmean(window, axis=0, keepdims=True)
            sigma = np.nanstd(window, axis=0, keepdims=True)
            sigma = np.where(sigma == 0, 1.0, sigma)
            features[idx] = (window - mu) / sigma
            np.nan_to_num(features[idx], copy=False, nan=0.0, posinf=0.0, neginf=0.0)

            labels[idx] = int(labs[i])
            symbols_out.append(sym)
            dates_out.append(dts[i])
            idx += 1

    return features[:idx], labels[:idx], symbols_out, dates_out


print("Create sliding windows")
train_features, train_labels, _, _ = create_windows(
    train_pdf, feature_cols, window_size, window_stride, price_limit_threshold
)
del train_pdf
print(f"  Train windows: {len(train_labels):,}")

val_features, val_labels, _, _ = create_windows(
    val_pdf, feature_cols, window_size, window_stride, price_limit_threshold
)
del val_pdf
print(f"  Val windows: {len(val_labels):,}")

test_features, test_labels, test_symbols, test_dates = create_windows(
    test_pdf, feature_cols, window_size, window_stride, price_limit_threshold
)
del test_pdf
print(f"  Test windows:  {len(test_labels):,}")

if len(train_features) == 0:
    raise ValueError("Training set is empty - check ml_features table has 'train' split data.")
if len(val_features) == 0:
    raise ValueError("Validation set is empty - check ml_features table has 'val' split data.")

print(f"\nWindow shape: {train_features.shape[1]} × {train_features.shape[2]}  (expected: {window_size} × {len(feature_cols)})")
print(f"NaN: {np.isnan(train_features).sum()}")
print(f"Inf: {np.isinf(train_features).sum()}")

#### 2. PyTorch dataset + class weights

In [0]:
class WyckoffDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.from_numpy(features)
        self.labels = torch.from_numpy(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


train_ds = WyckoffDataset(train_features, train_labels)
val_ds = WyckoffDataset(val_features, val_labels)
test_ds = WyckoffDataset(test_features, test_labels)

actual_window_size = train_ds.features.shape[1]
actual_num_features = train_ds.features.shape[2]

class_counts = np.bincount(train_ds.labels, minlength=number_of_phases)
total = class_counts.sum()
class_weights = total / (number_of_phases * class_counts.clip(min=1))
max_cw = CNNLSTMConfig().max_class_weight
class_weights = np.clip(class_weights, 1.0, max_cw)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

#### 3. Model with ablation 

In [0]:
class MultiScaleCNNBlock(nn.Module):
    def __init__(self, in_channels, filters, kernel_size):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, filters, kernel_size, padding=padding)
        self.bn1 = nn.BatchNorm1d(filters)
        self.conv2 = nn.Conv1d(filters, filters, kernel_size, padding=padding)
        self.bn2 = nn.BatchNorm1d(filters)

    def forward(self, x):
        x = Fn.relu(self.bn1(self.conv1(x)))
        x = Fn.relu(self.bn2(self.conv2(x)))
        return x


class WyckoffCNNLSTM(nn.Module):
    def __init__(self, config: CNNLSTMConfig, use_cnn: bool = True,
                 use_bilstm: bool = True, use_attention: bool = True):
        super().__init__()
        self.use_cnn = use_cnn
        self.use_bilstm = use_bilstm
        self.use_attention = use_attention

        if use_cnn:
            self.cnn_branches = nn.ModuleList([
                MultiScaleCNNBlock(config.num_features, filters, kernel)
                for filters, kernel in zip(config.cnn_filters, config.cnn_kernels)
            ])
            total_cnn_out = sum(config.cnn_filters)
            self.cnn_bn = nn.BatchNorm1d(total_cnn_out)
            self.cnn_dropout = nn.Dropout(config.cnn_dropout)
            post_cnn_dim = total_cnn_out
        else:
            post_cnn_dim = config.num_features

        if use_bilstm:
            self.lstm = nn.LSTM(
                input_size=post_cnn_dim,
                hidden_size=config.lstm_hidden,
                num_layers=config.lstm_layers,
                batch_first=True, bidirectional=True,
                dropout=config.lstm_dropout if config.lstm_layers > 1 else 0,
            )
            post_lstm_dim = config.lstm_hidden * 2
        else:
            post_lstm_dim = post_cnn_dim

        if use_attention:
            assert post_lstm_dim % config.attention_heads == 0, \
                f"post_lstm_dim={post_lstm_dim} not divisible by heads={config.attention_heads}"
            self.attention = nn.MultiheadAttention(
                embed_dim=post_lstm_dim,
                num_heads=config.attention_heads,
                batch_first=True, dropout=config.attention_dropout,
            )
            self.layer_norm = nn.LayerNorm(post_lstm_dim)

        self.classifier = nn.Sequential(
            nn.Linear(post_lstm_dim, config.mlp_hidden),
            nn.BatchNorm1d(config.mlp_hidden),
            nn.ReLU(),
            nn.Dropout(config.mlp_dropout),
            nn.Linear(config.mlp_hidden, config.num_classes),
        )

    def forward(self, x):
        if self.use_cnn:
            x_t = x.transpose(1, 2)
            cnn_outs = [branch(x_t) for branch in self.cnn_branches]
            x_cnn = torch.cat(cnn_outs, dim=1)
            x_cnn = self.cnn_dropout(self.cnn_bn(x_cnn))
            x = x_cnn.transpose(1, 2)

        if self.use_bilstm:
            x, _ = self.lstm(x)

        if self.use_attention:
            attn_out, _ = self.attention(x, x, x)
            x = self.layer_norm(attn_out + x)

        pooled = x.mean(dim=1)
        return self.classifier(pooled)

#### 4. Training loop

In [0]:
def train_model(model, train_ds, val_ds, config, class_weights, device, generator=None):
    train_loader = DataLoader(
        train_ds, batch_size=config.batch_size, shuffle=True, drop_last=True,generator=generator,
    )
    val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False)

    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=config.label_smoothing)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-6)

    best_val_f1 = 0.0
    patience_counter = 0
    best_state = None

    for epoch in range(1, config.max_epochs + 1):
        model.train()
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip_norm)
            optimizer.step()

        model.eval()
        val_loss = 0.0
        val_preds, val_lbls = [], []
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                logits = model(batch_x)
                val_loss += criterion(logits, batch_y).item() * batch_x.size(0)
                val_preds.extend(logits.argmax(dim=1).cpu().numpy())
                val_lbls.extend(batch_y.cpu().numpy())
        val_loss /= len(val_ds)
        val_f1 = f1_score(val_lbls, val_preds, average="macro", zero_division=0)
        scheduler.step(metrics=val_loss)

        mlflow.log_metrics({"val_loss": val_loss, "val_f1": val_f1}, step=epoch)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= config.early_stop_patience:
                print(f"  Early stop at epoch {epoch}, best val F1: {best_val_f1:.4f}")
                break

    if best_state:
        model.load_state_dict(best_state)
        model.to(device)

    return model, best_val_f1

#### 5. Run ablation

In [0]:
ablation_config = [
    {"name": "full",    "use_cnn": True,  "use_bilstm": True,  "use_attention": True},
    {"name": "no_cnn",  "use_cnn": False, "use_bilstm": True,  "use_attention": True},
    {"name": "no_lstm", "use_cnn": True,  "use_bilstm": False, "use_attention": True},
    {"name": "no_attn", "use_cnn": True,  "use_bilstm": True,  "use_attention": False},
]
ablation_seed = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model_config = CNNLSTMConfig()
model_config.num_features = actual_num_features
model_config.window_size = actual_window_size

mlflow.set_experiment(model_config.experiment_name)

ablation_rows = []
_parent_name = f"ablation_{pd.Timestamp.now().strftime('%Y%m%d_%H%M')}"
with mlflow.start_run(run_name=_parent_name) as parent_run:
    mlflow.log_params({
        "seed": ablation_seed,
        "n_variants": len(ablation_config),
        "train_samples": len(train_ds),
        "test_samples": len(test_ds),
        "augmentation": False,
    })

    for cfg in ablation_config:
        print(f"Variant: {cfg['name']}")

        np.random.seed(ablation_seed)
        torch.manual_seed(ablation_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(ablation_seed)
            torch.backends.cudnn.deterministic = True

        loader_generator = torch.Generator()
        loader_generator.manual_seed(ablation_seed)

        with mlflow.start_run(run_name=cfg["name"], nested=True) as child_run:
            mlflow.log_params({
                "variant": cfg["name"],
                "use_cnn": cfg["use_cnn"],
                "use_bilstm": cfg["use_bilstm"],
                "use_attention": cfg["use_attention"],
                "seed": ablation_seed,
            })

            model = WyckoffCNNLSTM(
                model_config,
                use_cnn=cfg["use_cnn"],
                use_bilstm=cfg["use_bilstm"],
                use_attention=cfg["use_attention"],
            ).to(device)
            n_params = sum(p.numel() for p in model.parameters())
            mlflow.log_param("params_count", n_params)
            print(f"  Params: {n_params:,}")

            model, best_val_f1 = train_model(
                model, train_ds, val_ds, model_config, class_weights_tensor, device,             generator=loader_generator,
            )

            test_loader = DataLoader(test_ds, batch_size=model_config.batch_size, shuffle=False)
            model.eval()
            preds, lbls = [], []
            with torch.no_grad():
                for bx, by in test_loader:
                    logits = model(bx.to(device))
                    preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
                    lbls.extend(by.numpy().tolist())

            f1_macro = f1_score(lbls, preds, average="macro", zero_division=0)
            f1_weighted = f1_score(lbls, preds, average="weighted", zero_division=0)
            acc = accuracy_score(lbls, preds)

            mlflow.log_metrics({
                "test_f1_macro": f1_macro,
                "test_f1_weighted": f1_weighted,
                "test_accuracy": acc,
                "best_val_f1": best_val_f1,
            })

            ablation_rows.append((
                cfg["name"],
                cfg["use_cnn"], cfg["use_bilstm"], cfg["use_attention"],
                ablation_seed,
                float(f1_macro), float(f1_weighted), float(acc), float(best_val_f1),
                int(n_params),
                datetime.now(),
                child_run.info.run_id,
            ))

            print(f"  Test macro F1: {f1_macro:.4f}  |  val F1: {best_val_f1:.4f}")

#### 6. Save results

In [0]:
ablation_schema = StructType([
    StructField("variant", StringType(), False),
    StructField("use_cnn", BooleanType()),
    StructField("use_bilstm", BooleanType()),
    StructField("use_attention", BooleanType()),
    StructField("seed", IntegerType()),
    StructField("macro_f1", DoubleType()),
    StructField("weighted_f1", DoubleType()),
    StructField("accuracy", DoubleType()),
    StructField("val_f1", DoubleType()),
    StructField("params_count", LongType()),
    StructField("eval_date", TimestampType()),
    StructField("run_id", StringType()),
])

abl_df = spark.createDataFrame(ablation_rows, schema=ablation_schema)
abl_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tables["component_contribution_results"])
print(f"Successfull written {len(ablation_rows)} rows to {tables['component_contribution_results']}")

#### 7. Summary

In [0]:
summary = pd.DataFrame(ablation_rows, columns=[
    "variant", "use_cnn", "use_bilstm", "use_attention", "seed",
    "macro_f1", "weighted_f1", "accuracy", "val_f1",
    "params_count", "eval_date", "run_id",
])[["variant", "macro_f1", "weighted_f1", "accuracy", "val_f1", "params_count"]]

print("Results")
print(summary.to_string(index=False, float_format="%.4f"))

full_f1 = summary[summary["variant"] == "full"]["macro_f1"].values[0]
print("\nMacro F1 delta vs 'full':")
for _, row in summary.iterrows():
    delta = row["macro_f1"] - full_f1
    print(f"  {row['variant']:>8s}: {delta:+.4f}")